# Demo: orquestador multi-agente

Este notebook muestra el **flujo de delegación** del orquestador: el supervisor recibe un pedido que requiere **investigación** (buscar reseñas y precios) y **análisis** (sentimiento y estadísticas), delega en cada especialista, valida los resultados con su rúbrica y, cuando todo está completo, pide la **síntesis** final.

Requiere `GOOGLE_API_KEY` en el `.env` (o `LLM_PROVIDER=openai` con `OPENAI_API_KEY`).

In [ ]:
import logging, warnings
from dotenv import find_dotenv, load_dotenv
from IPython.display import Image, Markdown, display

load_dotenv(find_dotenv(usecwd=True))   # the repo .env, searched from the notebook's folder up
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.basicConfig(level=logging.WARNING)

from graph import build_graph, run_task, save_trace
from main import DEFAULT_TASK

app = build_graph()

## 1. El grafo

Topología jerárquica: todos los especialistas vuelven al supervisor, que es el único que decide el siguiente paso o el final (`FINISH`).

In [ ]:
print(app.get_graph().draw_mermaid())
display(Image("graph.png"))

## 2. Ejecución

Cada bloque es un paso del grafo: 🧭 decisión del supervisor (con el estado de la rúbrica), 🔎 investigador, 📊 analista, ✍️ sintetizador.

In [ ]:
state = await run_task(app, DEFAULT_TASK)
save_trace(state, __import__("pathlib").Path("traces/notebook_run.json"))

## 3. Decisiones del supervisor

Qué eligió en cada paso, por qué, y si una regla del sistema corrigió al LLM (`overridden`).

In [ ]:
rows = ["| Paso | Rúbrica | Destino | Corregido | Motivo |", "|---|---|---|---|---|"]
for d in state["decisions"]:
    rubric = " ".join(("✓" if v else "✗") for v in d.rubric.values())
    rows.append(f"| {d.step} | {rubric} | **{d.next_agent}** | {'sí' if d.overridden else ''} | {d.reason} |")
display(Markdown("\n".join(rows)))

## 4. Quién aportó qué

Cada contribución queda registrada con el agente, la ronda (1 = primer intento, 2 = refinamiento), la instrucción recibida y las herramientas usadas.

In [ ]:
rows = ["| Agente | Ronda | Herramientas | Instrucción |", "|---|---|---|---|"]
for c in state["contributions"]:
    rows.append(f"| {c.agent} | {c.round} | {', '.join(c.tools_used) or '—'} | {c.instruction[:120]} |")
display(Markdown("\n".join(rows)))

## 5. Resultado del análisis (estructurado)

Los números salen de las herramientas, no del texto del LLM.

In [ ]:
rows = ["| Producto | Reseñas | Puntaje | Sentimiento | % positivas | Precio prom. | Más barato | Pros | Contras |",
        "|---|---|---|---|---|---|---|---|---|"]
for p in state["analysis"].products:
    rows.append(f"| {p.product} | {p.review_count} | {p.avg_rating} | {p.sentiment_score} | {p.positive_pct}% | "
                f"${p.avg_price:,.0f} | {p.best_store} (${p.min_price:,}) | {', '.join(p.pros)} | {', '.join(p.cons)} |")
display(Markdown("\n".join(rows)))

## 6. Respuesta final

In [ ]:
display(Markdown(state["final_answer"]))